# 2. Model inputs

Part 2 of 5. The model only sees data that is available without heat metering. Three input groups:
- Design and installation (14): known before the heat pump runs
- Electricity and weather (13): from daily whole-system electricity and outdoor temperature
- Combined (4): electricity relative to the home, for example kWh per m2

Notebook 3 compares three input sets: design only, electricity only, and all inputs. Runtime is under a minute.

## Step 1: Setup

In [ ]:
from google.colab import drive, files
drive.mount('/content/drive')

import os
DATA_DIR = '/content/drive/MyDrive/EoH_project'      # the raw EoH files
OUT_DIR = DATA_DIR + '/q1_paper'                     # everything this paper series makes
FIG_DIR = OUT_DIR + '/figures'
TABLE_DIR = OUT_DIR + '/tables'
os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(TABLE_DIR, exist_ok=True)

import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import rankdata

plt.rcParams.update({'font.size': 10, 'axes.titlesize': 11, 'axes.labelsize': 10, 'legend.fontsize': 8,
                     'axes.spines.top': False, 'axes.spines.right': False})
BLUE = '#2f6f9f'
ORANGE = '#c96a14'
GREY = '#8a8f98'
GREEN = '#3f8f5a'
MAKER = {'Daikin': 'Manufacturer A', 'Mitsubishi': 'Manufacturer B', 'Vaillant': 'Manufacturer C'}
MAKER_COLOR = {'Daikin': '#2f6f9f', 'Mitsubishi': '#8a5aa8', 'Vaillant': '#5b8c5a'}

print('ready')

## Step 2: Read notebook 1 outputs

`q1_homes.csv, q1_daily.csv`

In [ ]:
homes = pd.read_csv(OUT_DIR + '/q1_homes.csv')
daily = pd.read_csv(OUT_DIR + '/q1_daily.csv', parse_dates=['Date'])
print('homes:', len(homes), '| weak:', homes['weak'].sum(), f"({100 * homes['weak'].mean():.1f}%)")
print('days:', f'{len(daily):,}')

## Step 3: Design and installation inputs

From MCS design records and property data:

| Input | Meaning |
|---|---|
| `MCS_SCOP` | Design seasonal COP calculated by the installer |
| `MCS_Flow_Temp` | Design flow temperature (°C) |
| `MCS_SHLoad` | Design space-heating load on the coldest day (kW) |
| `MCS_Hloss` | Design heat loss of the home |
| `MCS_SHAnnual` | Design annual space-heating demand (kWh) |
| `MCS_DHWAnnual` | Design annual hot water demand (kWh) |
| `Total_Floor_Area` | Floor area (m2) |
| `HP_Size_kW` | Heat pump capacity (kW) |
| `oversize_ratio` | Capacity divided by design heat load |
| `TS_Water_Size` | Hot water cylinder volume |
| `No_Adults` | Number of adults |
| `is_high_temp` | High-temperature model (1) or standard (0) |
| `brand_Daikin`, `brand_Mitsubishi` | Manufacturer dummies (third manufacturer when both are 0) |

Missing values are filled later with training-fold medians.

In [ ]:
homes['is_high_temp'] = (homes['HP_Installed'] == 'HT_ASHP').astype(int)
homes['brand_Daikin'] = (homes['HP_Brand'] == 'Daikin').astype(int)
homes['brand_Mitsubishi'] = (homes['HP_Brand'] == 'Mitsubishi').astype(int)
homes['oversize_ratio'] = homes['HP_Size_kW'] / homes['MCS_SHLoad']
homes['design_heat_kWh'] = homes['MCS_SHAnnual'] + homes['MCS_DHWAnnual']

design_inputs = ['MCS_SCOP', 'MCS_Flow_Temp', 'MCS_SHLoad', 'MCS_Hloss', 'MCS_SHAnnual', 'MCS_DHWAnnual',
                 'Total_Floor_Area', 'HP_Size_kW', 'oversize_ratio', 'TS_Water_Size', 'No_Adults',
                 'is_high_temp', 'brand_Daikin', 'brand_Mitsubishi']

pd.DataFrame({'median': homes[design_inputs].median(),
              '% filled': 100 * homes[design_inputs].notna().mean()}).round(2)

## Step 4: Energy signature for each home

Heating degree days per day, with a 15.5 °C base:

`HDD = max(0, 15.5 - outside_temp)`

A straight line is fitted to daily electricity against HDD for each home:

`daily kWh = base + slope x HDD`

- slope: extra electricity per degree day. It depends on home size and insulation as well as heat pump efficiency.
- base: electricity on warm days, mostly hot water.
- fit (R2): share of the daily variation explained by weather.

Only valid days are used, with at least 150 per home. The plot shows the example home from notebook 1: cold days (below 3 °C) in blue, mild days (6 to 12 °C) in orange.

In [ ]:
daily['HDD'] = (15.5 - daily['outside_temp']).clip(lower=0)
daily['month'] = daily['Date'].dt.month
good = daily.dropna(subset=['elec_kWh', 'outside_temp']).copy()

rows = []
for pid, one in good.groupby('Property_ID'):
    if len(one) < 150:
        continue
    slope, base = np.polyfit(one['HDD'], one['elec_kWh'], 1)
    rows.append({'Property_ID': pid, 'days_used': len(one),
                 'sig_slope': slope, 'sig_base': base,
                 'sig_r2': np.corrcoef(one['HDD'], one['elec_kWh'])[0, 1] ** 2})
signature = pd.DataFrame(rows)
print('homes with an energy signature:', len(signature), 'of', len(homes))

with_data = homes[homes['coverage_%'] >= 90]
example = with_data.loc[(with_data['SPF'] - with_data['SPF'].median()).abs().idxmin(), 'Property_ID']
one = good[good['Property_ID'] == example]
s = signature[signature['Property_ID'] == example].iloc[0]
cold = one['outside_temp'] < 3
mild = (one['outside_temp'] >= 6) & (one['outside_temp'] < 12)
x = np.linspace(0, one['HDD'].max(), 50)

fig = plt.figure(figsize=(6, 4))
plt.scatter(one['HDD'], one['elec_kWh'], s=8, color=GREY, alpha=0.6, label='Other days')
plt.scatter(one.loc[cold, 'HDD'], one.loc[cold, 'elec_kWh'], s=10, color=BLUE, label='Cold days (below 3 C)')
plt.scatter(one.loc[mild, 'HDD'], one.loc[mild, 'elec_kWh'], s=10, color=ORANGE, label='Mild days (6 to 12 C)')
plt.plot(x, s['sig_base'] + s['sig_slope'] * x, color='red', linewidth=1.5, label='Fitted line')
plt.xlabel('Heating degree-days (how cold the day was)')
plt.ylabel('Whole-system electricity (kWh/day)')
plt.title(f"Energy signature, home {example} (slope {s['sig_slope']:.2f} kWh per HDD)")
plt.legend()
plt.show()

signature.describe().round(3)

# ---- save + download this figure (comment out the 2 lines below after the first run) ----
fig.savefig(FIG_DIR + '/q1_02_step04_energy_signature.png', dpi=300, bbox_inches='tight')
files.download(FIG_DIR + '/q1_02_step04_energy_signature.png')

## Step 5: Other electricity inputs

| Input | Meaning |
|---|---|
| `annual_kWh` | Annual electricity (mean of valid days x 365) |
| `summer_kWh_per_day` | Mean daily electricity, June to August |
| `winter_kWh_per_day` | Mean daily electricity, December to February |
| `base_share` | Summer daily use x 365 as a share of annual use (roughly hot water and base load) |
| `kWh_per_HDD_cold` | Electricity above base per degree day on cold days (below 3 °C) |
| `kWh_per_HDD_mild` | Same, on mild days (6 to 12 °C) |
| `cold_mild_ratio` | Ratio of the two; high values mean much more electricity in cold weather |
| `day_to_day_variation` | Standard deviation of daily electricity divided by its mean |
| `HDD_per_day`, `mean_outside_temp` | Weather in the same year for the same home |

At least 5 cold or mild days are needed, otherwise the value is left missing.

In [ ]:
rows = []
for pid, one in good.groupby('Property_ID'):
    if len(one) < 150:
        continue
    base = signature.loc[signature['Property_ID'] == pid, 'sig_base'].iloc[0]
    cold = one[one['outside_temp'] < 3]
    mild = one[(one['outside_temp'] >= 6) & (one['outside_temp'] < 12)]
    row = {'Property_ID': pid,
           'annual_kWh': one['elec_kWh'].mean() * 365,
           'summer_kWh_per_day': one.loc[one['month'].isin([6, 7, 8]), 'elec_kWh'].mean(),
           'winter_kWh_per_day': one.loc[one['month'].isin([12, 1, 2]), 'elec_kWh'].mean(),
           'day_to_day_variation': one['elec_kWh'].std() / one['elec_kWh'].mean(),
           'HDD_per_day': one['HDD'].mean(),
           'mean_outside_temp': one['outside_temp'].mean(),
           'kWh_per_HDD_cold': np.nan,
           'kWh_per_HDD_mild': np.nan}
    if len(cold) >= 5:
        row['kWh_per_HDD_cold'] = (cold['elec_kWh'] - base).sum() / cold['HDD'].sum()
    if len(mild) >= 5:
        row['kWh_per_HDD_mild'] = (mild['elec_kWh'] - base).sum() / mild['HDD'].sum()
    rows.append(row)
use = pd.DataFrame(rows)
use['cold_mild_ratio'] = use['kWh_per_HDD_cold'] / use['kWh_per_HDD_mild']
use.loc[use['kWh_per_HDD_mild'] <= 0, 'cold_mild_ratio'] = np.nan

features = signature.merge(use, on='Property_ID')
features['base_share'] = features['summer_kWh_per_day'] * 365 / features['annual_kWh']

electricity_inputs = ['annual_kWh', 'sig_slope', 'sig_base', 'sig_r2', 'summer_kWh_per_day', 'winter_kWh_per_day',
                      'base_share', 'kWh_per_HDD_cold', 'kWh_per_HDD_mild', 'cold_mild_ratio', 'day_to_day_variation',
                      'HDD_per_day', 'mean_outside_temp']
features[electricity_inputs].describe().T.round(3)

## Step 6: Combined inputs

Electricity on its own mostly reflects how much heat a home needed, not how well the heat pump worked (shown in notebook 4). These inputs relate electricity to the home:

| Input | Meaning |
|---|---|
| `kWh_per_m2` | Annual electricity per m2 |
| `kWh_per_HP_kW` | Annual electricity per kW of heat pump capacity |
| `spf_proxy` | Design annual heat divided by measured annual electricity |
| `spf_proxy_slope` | Design heat load divided by the energy-signature slope |

For the last one, the design load is assumed to cover a temperature difference of about 24 K (21 °C inside, -3 °C outside), so design heat per degree day is close to the design load itself:

`spf_proxy_slope = MCS_SHLoad x 24 / 24 / sig_slope = MCS_SHLoad / sig_slope`

In [ ]:
table = homes.merge(features, on='Property_ID', how='left')
table['kWh_per_m2'] = table['annual_kWh'] / table['Total_Floor_Area']
table['kWh_per_HP_kW'] = table['annual_kWh'] / table['HP_Size_kW']
table['spf_proxy'] = table['design_heat_kWh'] / table['annual_kWh']
table['spf_proxy_slope'] = table['MCS_SHLoad'] / table['sig_slope']
table.loc[table['sig_slope'] <= 0, 'spf_proxy_slope'] = np.nan

combined_inputs = ['kWh_per_m2', 'kWh_per_HP_kW', 'spf_proxy', 'spf_proxy_slope']
table[combined_inputs].describe().T.round(3)

## Step 7: Three input sets

- `design`: design and installation only (14)
- `electricity`: electricity and weather only (13)
- `both`: all inputs including the combined ones (31), used by the main model

In [ ]:
input_sets = {'design': design_inputs,
              'electricity': electricity_inputs,
              'both': design_inputs + electricity_inputs + combined_inputs}

pd.DataFrame({'inputs': {k: len(v) for k, v in input_sets.items()},
              '% empty cells': {k: round(100 * table[v].isna().mean().mean(), 2) for k, v in input_sets.items()}})

## Step 8: A first look before any model

Spearman rank correlation of each input with measured SPF, strongest first. Negative means higher values go with lower SPF; values near zero mean the input says little on its own.

In [ ]:
def spearman(a, b):
    ok = ~(pd.isna(a) | pd.isna(b))
    return np.corrcoef(rankdata(np.asarray(a)[ok]), rankdata(np.asarray(b)[ok]))[0, 1]


rows = []
for group, inputs in [('design', design_inputs), ('electricity', electricity_inputs), ('combined', combined_inputs)]:
    for c in inputs:
        rows.append({'input': c, 'group': group, 'rank corr with SPF': spearman(table[c], table['SPF'])})
first_look = pd.DataFrame(rows)
first_look['strength'] = first_look['rank corr with SPF'].abs()
first_look = first_look.sort_values('strength', ascending=False).drop(columns='strength').set_index('input')
first_look.round(3)

Four inputs against measured SPF, coloured by manufacturer. The red dashed line is the 2.5 threshold and each title shows the rank correlation.

(a) annual electricity, the raw number any meter sees
(b) electricity per m2
(c) SPF proxy
(d) design SCOP, for comparison

In [ ]:
show = [('annual_kWh', 'Annual electricity (kWh)'),
        ('kWh_per_m2', 'Electricity per floor area (kWh/m2)'),
        ('spf_proxy', 'Design heat / measured electricity'),
        ('MCS_SCOP', 'Design SCOP')]

fig = plt.figure(figsize=(10, 7))
for k, (col, label) in enumerate(show):
    plt.subplot(2, 2, k + 1)
    for brand in ['Daikin', 'Mitsubishi', 'Vaillant']:
        part = table[table['HP_Brand'] == brand]
        plt.scatter(part[col], part['SPF'], s=8, alpha=0.6, color=MAKER_COLOR[brand], label=MAKER[brand])
    plt.axhline(2.5, color='red', linestyle=':', linewidth=1)
    plt.xlabel(label)
    plt.ylabel('Measured SPF')
    plt.title(f"({'abcd'[k]}) rank corr = {spearman(table[col], table['SPF']):.2f}")
    if k == 0:
        plt.legend(loc='upper right')
plt.tight_layout()
plt.show()

# ---- save + download this figure (comment out the 2 lines below after the first run) ----
fig.savefig(FIG_DIR + '/q1_02_step08_inputs_vs_spf.png', dpi=300, bbox_inches='tight')
files.download(FIG_DIR + '/q1_02_step08_inputs_vs_spf.png')

## Step 9: Save for notebook 3

`q1_table.csv, q1_input_sets.json`

In [ ]:
table.to_csv(OUT_DIR + '/q1_table.csv', index=False)
with open(OUT_DIR + '/q1_input_sets.json', 'w') as f:
    json.dump(input_sets, f, indent=1)
first_look.to_csv(TABLE_DIR + '/q1_02_rank_corr_each_input.csv')
print('saved q1_table.csv', table.shape, '| q1_input_sets.json | q1_02_rank_corr_each_input.csv')